In [2]:
# Week 3 - Regression Stuff
# trying forward/backward selection, PCR, PLSR

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [3]:
# load the dataset
data = pd.read_csv("hypertension_dataset.csv")

# just checking what it looks like
data.head()

,Country,Age,BMI,Cholesterol,Systolic_BP,Diastolic_BP,Smoking_Status,Alcohol_Intake,Physical_Activity_Level,Family_History,...,Sleep_Duration,Heart_Rate,LDL,HDL,Triglycerides,Glucose,Gender,Education_Level,Employment_Status,Hypertension
0,UK,58,29.5,230,160,79,Never,27.9,Low,Yes,...,6.1,80,100,75,72,179,Female,Primary,Unemployed,High
1,Spain,34,36.2,201,120,84,Never,27.5,High,Yes,...,9.8,56,77,47,90,113,Male,Secondary,Unemployed,High
2,Indonesia,73,18.2,173,156,60,Current,1.8,High,Yes,...,5.2,75,162,56,81,101,Male,Primary,Employed,Low
3,Canada,60,20.3,183,122,94,Never,11.6,Moderate,Yes,...,7.5,71,164,93,94,199,Female,Secondary,Retired,High
4,France,73,21.8,296,91,97,Never,29.1,Moderate,Yes,...,5.0,52,108,74,226,157,Female,Primary,Employed,High


In [5]:
# pick target and features
# NOTE: change 'target_column' to the actual column we want to predict
X = data.drop(columns=['Hypertension'])
y = data['Hypertension']

# Convert target into 0/1
y = y.map({'Low': 0, 'High': 1})

In [8]:
# Make sure everything is numeric
X_encoded = pd.get_dummies(X, drop_first=True)

# Convert to float so statsmodels is happy
X_encoded = X_encoded.astype(float)

def forward_selection(X, y, significance_level=0.05):
    initial_features = []
    selected = list(initial_features)
    remaining = list(X.columns)

    while remaining:
        new_pvals = pd.Series(index=remaining, dtype=float)
        for col in remaining:
            model = sm.OLS(y, sm.add_constant(X[selected + [col]])).fit()
            new_pvals[col] = model.pvalues[col]

        min_pval = new_pvals.min()
        if min_pval < significance_level:
            best_feature = new_pvals.idxmin()
            selected.append(best_feature)
            remaining.remove(best_feature)
        else:
            break
    return selected

forward_feats = forward_selection(X_encoded, y)
print("Forward selected features:", forward_feats)



Forward selected features: []


In [ ]:
# backward selection (start with everything and drop high p-values)
def backward_selection(X, y, significance_level=0.05):
    features = list(X.columns)
    while len(features) > 0:
        model = sm.OLS(y, sm.add_constant(X[features])).fit()
        pvals = model.pvalues.iloc[1:]  # skip intercept
        if pvals.max() > significance_level:
            worst = pvals.idxmax()
            features.remove(worst)
        else:
            break
    return features

backward_feats = backward_selection(X, y)
print("Backward selected features:", backward_feats)

In [ ]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale the data (important for PCA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# just try with 5 components (can change later)
k = 5
pcr = LinearRegression()
pcr.fit(X_train_pca[:, :k], y_train)

y_pred_pcr = pcr.predict(X_test_pca[:, :k])

print("PCR R2:", r2_score(y_test, y_pred_pcr))
print("PCR MSE:", mean_squared_error(y_test, y_pred_pcr))

In [ ]:
# PLSR also needs scaling
n_comp = 5  # number of components to try
pls = PLSRegression(n_components=n_comp)
pls.fit(X_train_scaled, y_train)

y_pred_pls = pls.predict(X_test_scaled)

print("PLSR R2:", r2_score(y_test, y_pred_pls))
print("PLSR MSE:", mean_squared_error(y_test, y_pred_pls))

In [ ]:
# just to see how much variance each PC explains
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.show()